In [14]:
# импорты библиотек
import gspread
import pandas as pd
import numpy as np
from datetime import datetime, date
from google.oauth2.service_account import Credentials
import re

In [15]:

# Константы подключения
SPREADSHEET_ID = "1wxNIhrPjoFQf3vqatu9oK78W5RmNoI3YG6sf9i4GWAQ"

# Основные листы + листы "Уволенные" и "Стаффинг" (для полноты данных)
SHEET_NAMES = [
    "DEMO Лист", "A-Benefits", "1C", "Active Bank", "A-Sber", 
    "Betta Bank", "Evergreen Bank+Laba ABS", "Liberty Bank", 
    "Investment", "L&D", "MyCare", "SQL Trainee", "ProBusiness",
    "Уволенные 2026", "Стаффинг"
]

# Отдельный лист для промежуточных статусов
INTERIM_STATUS_SHEET = "Промежуточный статус стажера"

# Индексы столбцов для основных листов (начинаются с 0)
COL_TECH = 1      # B
COL_TECHLEAD = 2  # C
COL_NAME = 3      # D
COL_START = 5     # F
COL_STOP = 6      # G
COL_REASON = 7    # H

# Индексы столбцов для листа "Промежуточный статус стажера"
# Структура: B=Проект, C=Технология, D=Техлид, E=ФИО, F=Зачетка, G=Дата старта, H=Дата стопа, I=Причина ухода
INTERIM_COL_TECH = 2         # C
INTERIM_COL_TECHLEAD = 3     # D
INTERIM_COL_NAME = 4         # E
INTERIM_COL_START = 6        # G
INTERIM_COL_STOP = 7         # H
INTERIM_COL_REASON = 8       # I

# Настройки отчетного периода
REPORT_YEAR = 2026
REPORT_MONTH = 10  # Октябрь

In [16]:
def authenticate_with_service_account(credentials_file: str) -> gspread.Client:
    """Аутентификация через Service Account"""
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
    creds = Credentials.from_service_account_file(credentials_file, scopes=scopes)
    return gspread.authorize(creds)

def clean_date(val) -> date | None:
    """Преобразует значение в объект даты
    """
    if pd.isna(val) or str(val).strip() == '':
        return None
    if isinstance(val, datetime):
        return val.date()
    if isinstance(val, date):
        return val
    
    val_str = str(val).strip()
    for fmt in ['%d.%m.%Y', '%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%dT%H:%M:%S']:
        try:
            return datetime.strptime(val_str, fmt).date()
        except ValueError:
            continue
    return None

def parse_date_range(val) -> tuple[date | None, date | None]:
    """
    Парсит дату или диапазон дат 
    Возвращает (start_date, end_date).
    """
    if pd.isna(val) or str(val).strip() == '':
        return None, None
    
    val_str = str(val).strip()
    
    # Проверяем диапазон (дефис между датами)
    if '-' in val_str and '.' in val_str:
        parts = val_str.split('-')
        if len(parts) == 2:
            start_date = clean_date(parts[0].strip())
            end_date = clean_date(parts[1].strip())
            return start_date, end_date
    
    # Обычная дата
    return clean_date(val_str), None

def clean_fio(val) -> str:
    """Очищает ФИО от лишних символов
    """
    if pd.isna(val) or str(val).strip() == '':
        return ''
    val_str = str(val).strip()
    match = re.match(r'^[A-Za-zА-Яа-яЁё\s\-]+', val_str)
    if match:
        return match.group(0).strip()
    return val_str

def normalize_fio(fio: str) -> str:
    """Нормализует ФИО для корректного подсчета уникальных людей
    """
    if not fio:
        return ''
    return re.sub(r'\s+', ' ', fio.lower().strip()).title()

def clean_tech_name(tech_name):
    """
    Нормализует название технологии:
    - Приводит к верхнему регистру
    - Заменяет кириллические символы на латинские
    - Удаляет спецсимволы и пробелы
    - Объединяет синонимы (QA, AQA -> QAJAVA)
    """
    if pd.isna(tech_name):
        return tech_name
    
    tech_str = str(tech_name).upper()
    
    # 1. Замена кириллических символов на латинские
    cyrillic_to_latin = str.maketrans({
        'А': 'A', 'В': 'B', 'Е': 'E', 'К': 'K', 'М': 'M',
        'Н': 'H', 'О': 'O', 'Р': 'P', 'С': 'C', 'Т': 'T', 'Х': 'X',
    })
    tech_str = tech_str.translate(cyrillic_to_latin)
    
    # 2. Оставляем только буквы и цифры
    tech_str = re.sub(r'[^A-Z0-9]', '', tech_str)
    
    # 3. Маппинг синонимов (Исправление: теперь это QAJAVA)
    if tech_str in ['QA', 'AQA', 'QAJAVA', 'AQAJAVA', 'QAJAV', 'AQJAV', 'QAENGINER']:
        tech_str = 'QAJAVA'
        
    # Дополнительные частые нормализации для 1С и других
    if tech_str in ['1CDEV', '1C']: tech_str = '1C DEV'
    if tech_str in ['1CBA', '1CANALYTIC']: tech_str = '1C BA'
    if tech_str == 'DATAANALYST': tech_str = 'DATA ANALYST'
    if tech_str == 'DATAENGINEER': tech_str = 'DATA ENGINEER'

    return tech_str

def is_valid_techlead(name: str) -> bool:
    """Проверяет, что имя техлида выглядит как реальное 'Имя Фамилия'"""
    if pd.isna(name):
        return False
    name = str(name).strip()
    if len(name) < 3:
        return False
    if " " not in name:
        return False
    blacklist = ["-", "х", "x", "2-я технология", "вторая технология", "команда", "techleads"]
    if name.lower() in blacklist:
        return False
    return True

def extract_sheet_data(worksheet) -> pd.DataFrame:
    """Парсит основные листы гугл дока"""
    values = worksheet.get_values()
    if not values:
        return pd.DataFrame()
    
    header_idx = -1
    for i, row in enumerate(values):
        if len(row) > COL_NAME:
            col_b = str(row[COL_TECH]).strip().lower()
            col_c = str(row[COL_TECHLEAD]).strip().lower()
            col_d = str(row[COL_NAME]).strip().lower()
            
            is_tech = 'технология' in col_b
            is_techlead = 'техлид' in col_c or 'techlead' in col_c
            is_name = 'фамилия' in col_d or 'фио' in col_d
            
            if is_tech and is_techlead and is_name:
                header_idx = i
                break
    
    if header_idx == -1:
        return pd.DataFrame()

    data = []
    for i in range(header_idx + 1, len(values)):
        row = values[i]
        while len(row) <= COL_REASON:
            row.append('')
        
        col_b_val = str(row[COL_TECH]).strip()
        if not col_b_val:
            continue
            
        fio = clean_fio(row[COL_NAME]).strip()
        if not fio:
            continue

        data.append({
            'Технология': col_b_val,
            'Техлид': clean_fio(row[COL_TECHLEAD]),
            'Лаборант': fio,
            'Дата старта': clean_date(row[COL_START]),
            'Дата стопа': clean_date(row[COL_STOP]),
            'Причина_статус': str(row[COL_REASON]).strip().lower(),
            'Источник': worksheet.title
        })
    
    return pd.DataFrame(data)

def extract_interim_status_data(worksheet) -> pd.DataFrame:
    """
    Парсит лист 'Промежуточный статус стажера'.
    Извлекает только записи с 'Неоплачиваемый отпуск'.
    """
    values = worksheet.get_values()
    if not values:
        return pd.DataFrame()
    
    header_idx = -1
    for i, row in enumerate(values):
        if len(row) > INTERIM_COL_NAME:
            col_c = str(row[INTERIM_COL_TECH]).strip().lower()
            col_d = str(row[INTERIM_COL_TECHLEAD]).strip().lower()
            col_e = str(row[INTERIM_COL_NAME]).strip().lower()
            
            is_tech = 'технология' in col_c
            is_techlead = 'техлид' in col_d or 'techlead' in col_d
            is_name = 'фамилия' in col_e or 'фио' in col_e
            
            if is_tech and is_techlead and is_name:
                header_idx = i
                break
    
    if header_idx == -1:
        return pd.DataFrame()

    data = []
    for i in range(header_idx + 1, len(values)):
        row = values[i]
        while len(row) <= INTERIM_COL_REASON:
            row.append('')
        
        col_c_val = str(row[INTERIM_COL_TECH]).strip()
        if not col_c_val:
            continue
            
        fio = clean_fio(row[INTERIM_COL_NAME]).strip()
        if not fio:
            continue

        reason = str(row[INTERIM_COL_REASON]).strip().lower()
        
        # Нас интересует только "Неоплачиваемый отпуск"
        if 'неоплачиваемый отпуск' in reason:
            # Парсим диапазон дат
            start_date, end_date = parse_date_range(row[INTERIM_COL_STOP])
            
            data.append({
                'Технология': col_c_val,
                'Техлид': clean_fio(row[INTERIM_COL_TECHLEAD]),
                'Лаборант': fio,
                'Дата старта': start_date if start_date else clean_date(row[INTERIM_COL_START]),
                'Дата стопа': end_date,
                'Причина_статус': reason,
                'Источник': worksheet.title
            })
    
    return pd.DataFrame(data)

In [17]:
# генерация отчета

def generate_payroll_reports(raw_df, interim_df, year, month):
    df = raw_df.copy()
    
    # 1. Нормализация технологий
    df['Технология'] = df['Технология'].apply(clean_tech_name)
    
    # 2. Очистка и фильтрация техлидов
    df['Техлид'] = df['Техлид'].apply(clean_fio)
    df = df[df['Техлид'].apply(is_valid_techlead)].copy()
    
    # 3. Нормализация ФИО
    df['Лаборант_норм'] = df['Лаборант'].apply(normalize_fio)
    
    # 4. Конвертация дат
    df['Дата старта'] = pd.to_datetime(df['Дата старта'], errors='coerce')
    df['Дата стопа'] = pd.to_datetime(df['Дата стопа'], errors='coerce')
    df = df.dropna(subset=['Дата старта'])
    
    # 5. ДЕДУПЛИКАЦИЯ (Решение задачи №1)
    # Если человек есть и на основном листе, и в "Уволенных", мы должны взять запись с датой стопа.
    # Сортируем так, чтобы записи с "Дата стопа" были первыми, затем по дате старта
    df = df.sort_values(by=['Дата стопа', 'Дата старта'], ascending=[False, False], na_position='last')
    # Удаляем дубликаты, оставляя первую (самую актуальную/завершенную) запись
    df = df.drop_duplicates(subset=['Лаборант_норм', 'Технология', 'Техлид'], keep='first')
    
    # 6. Обработка неоплачиваемых отпусков
    unpaid_leave_periods = []
    if interim_df is not None and not interim_df.empty:
        interim_df['Дата старта'] = pd.to_datetime(interim_df['Дата старта'], errors='coerce')
        interim_df['Дата стопа'] = pd.to_datetime(interim_df['Дата стопа'], errors='coerce')
        interim_df['Лаборант_норм'] = interim_df['Лаборант'].apply(normalize_fio)
        interim_df['Технология'] = interim_df['Технология'].apply(clean_tech_name)
        interim_df['Техлид'] = interim_df['Техлид'].apply(clean_fio)
        
        for _, row in interim_df.iterrows():
            if pd.notna(row['Дата старта']) and pd.notna(row['Дата стопа']):
                unpaid_leave_periods.append({
                    'Лаборант_норм': row['Лаборант_норм'],
                    'Технология': row['Технология'],
                    'Техлид': row['Техлид'],
                    'start': row['Дата старта'],
                    'end': row['Дата стопа']
                })
    
    # 7. Генерация диапазона ТОЛЬКО РАБОЧИХ дней (Пн-Пт)
    start_date = pd.Timestamp(year=year, month=month, day=1)
    end_date = start_date + pd.DateOffset(months=1) - pd.DateOffset(days=1)
    
    all_days = pd.date_range(start_date, end_date)
    working_days = all_days[all_days.dayofweek < 5] # 0=Пн, 4=Пт
    
    total_working_days_in_month = len(working_days)
    
    daily_records = []
    
    # 8. Подсчет уникальных лаборантов для каждого РАБОЧЕГО дня
    for current_date in working_days:
        mask = (df['Дата старта'] <= current_date) & \
               ((df['Дата стопа'] >= current_date) | df['Дата стопа'].isna())
        
        active_df = df[mask].copy()
        
        # Исключаем неоплачиваемые отпуска
        if unpaid_leave_periods:
            for period in unpaid_leave_periods:
                if period['start'] <= current_date <= period['end']:
                    exclude_mask = (
                        (active_df['Лаборант_норм'] == period['Лаборант_норм']) &
                        (active_df['Технология'] == period['Технология']) &
                        (active_df['Техлид'] == period['Техлид'])
                    )
                    active_df = active_df[~exclude_mask]
        
        if not active_df.empty:
            counts = active_df.groupby(['Технология', 'Техлид'])['Лаборант_норм'].nunique().reset_index(name='количество лаборантов')
            counts['дата'] = current_date.date()
            daily_records.append(counts)
            
    if daily_records:
        daily_df = pd.concat(daily_records, ignore_index=True)
        daily_df = daily_df[['Технология', 'Техлид', 'дата', 'количество лаборантов']]
        daily_df = daily_df.sort_values(by=['дата', 'Технология', 'Техлид']).reset_index(drop=True)
    else:
        daily_df = pd.DataFrame(columns=['Технология', 'Техлид', 'дата', 'количество лаборантов'])
        
    # 9. Сводка за месяц и расчет оплаты
    summary_df = daily_df.groupby(['Технология', 'Техлид']).agg(
        отработано_рабочих_дней=('дата', 'count'),
        всего_лаборанто_дней=('количество лаборантов', 'sum')
    ).reset_index()
    
    # Справочник рейтов (костыльный хардкод из файла "Копия ФИДБЕКИ Aston.xlsx")
    RATES = {
        'BA/SA': 25000, 'BASА': 25000, 
        'JAVA': 22000, '1C DEV': 22000, '1C': 22000, 'PYTHON': 22000, 'ANDROID': 22000,
        'REACT': 17250,
        'DEVOPS': 25000, 'SRE': 25000,
        'QAJAVA': 23000, 'QA ENGINEER': 23000,
        'АРХИТЕКТОР': 27000,
        'DATA ANALYST': 22000, 'DATA ENGINEER': 25000,
        'UI/UX': 22000
    }
    
    def get_rate(tech):
        tech_clean = re.sub(r'[^A-Z0-9]', '', str(tech).upper())
        for key, val in RATES.items():
            if re.sub(r'[^A-Z0-9]', '', key) == tech_clean:
                return val
        return 22000 # Дефолтный рейт, если технология не найдена

    summary_df['Рейт'] = summary_df['Технология'].apply(get_rate)
    summary_df['Рабочих дней в месяце'] = total_working_days_in_month
    
    # Расчет: (Рейт / Рабочих дней) * Отработанных дней
    # Примечание: 'всего_лаборанто_дней' уже учитывает умножение на кол-во человек за каждый день
    summary_df['Дневной рейт'] = (summary_df['Рейт'] / total_working_days_in_month).round(2)
    summary_df['Итого к оплате'] = (summary_df['Дневной рейт'] * summary_df['всего_лаборанто_дней']).round(2)
    
    # Среднее количество (для справки)
    summary_df['среднее_лаборантов_в_день'] = (
        summary_df['всего_лаборанто_дней'] / max(total_working_days_in_month, 1)
    ).round(2)
    
    # Финальная сортировка
    summary_df = summary_df.sort_values(by=['Технология', 'Техлид']).reset_index(drop=True)
    
    # Переименуем колонки для красоты в итоговом Excel
    summary_df.rename(columns={
        'отработано_рабочих_дней': 'Отработано рабочих дней (на 1 чел)',
        'всего_лаборанто_дней': 'Сумма человеко-дней'
    }, inplace=True)
    
    return daily_df, summary_df

In [18]:
def main():
    print("Аутентификация через Service Account...")
    try:
        client = authenticate_with_service_account('credentials.json')
        print("Аутентификация успешна")
    except Exception as e:
        print(f"Ошибка аутентификации: {e}")
        return

    print("Открытие таблицы...")
    try:
        spreadsheet = client.open_by_key(SPREADSHEET_ID)
        print(f"Таблица открыта: {spreadsheet.title}")
    except Exception as e:
        print(f"ОШИБКА открытия таблицы: {e}")
        return

    # Парсинг основных листов (включая Уволенные и Стаффинг)
    all_data = []
    for sheet_name in SHEET_NAMES:
        try:
            worksheet = spreadsheet.worksheet(sheet_name)
            df = extract_sheet_data(worksheet)
            if not df.empty:
                all_data.append(df)
                print(f"  Лист '{sheet_name}': {len(df)} строк")
        except Exception as e:
            print(f"  Ошибка на листе {sheet_name}: {e}")

    # Парсинг листа "Промежуточный статус стажера"
    interim_df = pd.DataFrame()
    try:
        worksheet = spreadsheet.worksheet(INTERIM_STATUS_SHEET)
        interim_df = extract_interim_status_data(worksheet)
        if not interim_df.empty:
            print(f"  Лист '{INTERIM_STATUS_SHEET}': {len(interim_df)} записей о неоплачиваемых отпусках")
        else:
            print(f"  Лист '{INTERIM_STATUS_SHEET}': неоплачиваемых отпусков не найдено")
    except Exception as e:
        print(f"  Ошибка на листе {INTERIM_STATUS_SHEET}: {e}")

    if not all_data:
        print("ОШИБКА: Нет данных для сохранения.")
        return

    raw_df = pd.concat(all_data, ignore_index=True)
    print(f"\nУспешно спарсено {len(raw_df)} строк до фильтрации.")

    # Генерация отчетов
    daily_df, summary_df = generate_payroll_reports(raw_df, interim_df, REPORT_YEAR, REPORT_MONTH)
    
    print(f"После фильтрации артефактов в расчете участвует уникальных техлидов: {daily_df['Техлид'].nunique()}")

    filename = f'payroll_report_{REPORT_YEAR}_{REPORT_MONTH:02d}.xlsx'
    
    with pd.ExcelWriter(filename, engine='openpyxl', datetime_format='YYYY-MM-DD') as writer:
        # Лист 1: Ежедневный отчет
        daily_df.to_excel(writer, sheet_name='Ежедневный_отчет', index=False)
        # Лист 2: Сводка за месяц
        summary_df.to_excel(writer, sheet_name='Сводка_за_месяц', index=False)
        
        # Автоширина колонок
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            for i, col in enumerate(worksheet.columns):
                max_len = max(len(str(cell.value)) for cell in col if cell.value is not None)
                worksheet.column_dimensions[chr(65 + i)].width = min(max_len + 2, 30)

    print(f"\nОтчет успешно сохранен в файл: {filename}")
    print(f" - Лист 'Ежедневный_отчет': {len(daily_df)} строк (все дни месяца)")
    print(f" - Лист 'Сводка_за_месяц': {len(summary_df)} строк (агрегация для оплаты)")

if __name__ == "__main__":
    main()

Аутентификация через Service Account...
Аутентификация успешна
Открытие таблицы...
Таблица открыта: Копия ФИДБЕКИ Aston
  Лист 'DEMO Лист': 16 строк
  Лист 'A-Benefits': 16 строк
  Лист '1C': 43 строк
  Лист 'Active Bank': 51 строк
  Лист 'A-Sber': 30 строк
  Лист 'Betta Bank': 30 строк
  Лист 'Evergreen Bank+Laba ABS': 52 строк
  Лист 'Liberty Bank': 57 строк
  Лист 'Investment': 46 строк
  Лист 'L&D': 11 строк
  Лист 'MyCare': 27 строк
  Лист 'SQL Trainee': 22 строк
  Лист 'ProBusiness': 21 строк
  Лист 'Промежуточный статус стажера': 1 записей о неоплачиваемых отпусках

Успешно спарсено 422 строк до фильтрации.
После фильтрации артефактов в расчете участвует уникальных техлидов: 91

Отчет успешно сохранен в файл: payroll_report_2026_10.xlsx
 - Лист 'Ежедневный_отчет': 2002 строк (все дни месяца)
 - Лист 'Сводка_за_месяц': 91 строк (агрегация для оплаты)
